# 06 - Land / Subdivision LGD Model

**No-Income Land and Subdivision Exposure - Australian Bank-Style LGD Framework**

This module targets land/subdivision exposures where recovery depends on sale liquidity and planning status rather than operating income.

| Step | Description |
|---|---|
| 1 | Generate synthetic land / subdivision default dataset |
| 2 | Build base LGD from land liquidity and planning drivers |
| 3 | Apply stronger downturn stress (longer sale horizon + deeper haircut) |
| 4 | Produce EAD-weighted outputs by segment and zoning stage |
| 5 | Export interview-ready tables and checks |


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(48)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 90)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)



## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** transparent proxy assumptions are used where observed internal land workout tape is unavailable.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status (standard policy):** this is a portfolio-project proxy baseline, **not** internally calibrated to real workout outcomes.
- **Output standard:** report `lgd_base`, `lgd_downturn`, `lgd_final`, EAD-weighted outputs, and base recovery-time metric (`time_to_sell_base`).


## Objective
Build an interview-ready land/subdivision LGD module for low-income-carry collateral with weighted outputs.

## Drivers
- Zoning/stage risk
- Liquidity risk and market depth
- Value haircut
- Time to sell and holding costs

## Logic
- Base/downturn/final LGD with longer-horizon recovery modelling and weighted aggregation

## Downturn
- Stronger haircut, longer disposal horizon, and higher carry/discount impacts

## Validation
- Weighted scenario/segment outputs and validation checks

## Limitations
- Synthetic portfolio and proxy assumptions; future calibration requires internal workout data


## Why Land / Subdivision Differs from CRE and Development

- **Versus CRE investment:** land has no stabilised rental income, so DSCR/WALE-led recovery logic is less relevant.
- **Versus development finance:** land/subdivision is pre-income and often pre-completion; key risk is zoning/stage and market depth, not cost-to-complete execution alone.
- Recovery is mainly a liquidation path with longer sale periods and higher value haircut sensitivity.


## 1. Generate Synthetic Land / Subdivision Dataset


In [ ]:
n_cases = 220
rng = np.random.default_rng(48)

segments = ['infill_land', 'broadacre_subdivision', 'regional_land', 'masterplan_stage']
segment_weights = [0.28, 0.33, 0.22, 0.17]
zoning_stages = ['titled', 'da_approved', 'rezoning_pending', 'raw_land']

segment_cfg = {
    'infill_land': {'value_low': 8e6, 'value_high': 70e6, 'haircut': (0.10, 0.24), 'base_months': (8, 22)},
    'broadacre_subdivision': {'value_low': 20e6, 'value_high': 160e6, 'haircut': (0.12, 0.28), 'base_months': (12, 30)},
    'regional_land': {'value_low': 6e6, 'value_high': 65e6, 'haircut': (0.14, 0.34), 'base_months': (14, 36)},
    'masterplan_stage': {'value_low': 25e6, 'value_high': 220e6, 'haircut': (0.11, 0.30), 'base_months': (10, 32)},
}

stage_map = {'titled': 0.00, 'da_approved': 0.03, 'rezoning_pending': 0.08, 'raw_land': 0.12}

rows = []
for i in range(1, n_cases + 1):
    seg = rng.choice(segments, p=segment_weights)
    cfg = segment_cfg[seg]
    zoning_stage = rng.choice(zoning_stages, p=[0.26, 0.33, 0.24, 0.17])

    gross_land_value = float(rng.uniform(cfg['value_low'], cfg['value_high']))
    leverage = float(np.clip(rng.normal(0.70, 0.09), 0.45, 0.90))
    ead = gross_land_value * leverage * rng.uniform(0.98, 1.03)

    base_haircut = float(rng.uniform(cfg['haircut'][0], cfg['haircut'][1]))
    zoning_addon = stage_map[zoning_stage]
    liquidity_risk = float(np.clip(rng.normal(0.45, 0.20), 0.05, 0.95))
    market_depth = float(np.clip(rng.normal(0.50, 0.22), 0.05, 0.95))

    base_months = float(rng.uniform(cfg['base_months'][0], cfg['base_months'][1]))
    time_to_sell_months = float(np.clip(base_months * (1 + 0.65 * liquidity_risk + 0.50 * (1 - market_depth)), 6, 54))

    holding_cost_rate_pa = float(np.clip(rng.normal(0.065, 0.015), 0.03, 0.12))
    selling_cost_rate = float(np.clip(rng.normal(0.025, 0.006), 0.01, 0.05))

    contract_rate_proxy = float(np.clip(rng.normal(0.075, 0.012), 0.045, 0.115))
    cost_of_funds_proxy = float(np.clip(rng.normal(0.056, 0.010), 0.035, 0.090))
    discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)

    rows.append({
        'case_id': f'LAND{i:04d}',
        'segment': seg,
        'no_income_asset_flag': 1,
        'zoning_stage': zoning_stage,
        'gross_land_value': gross_land_value,
        'ead': ead,
        'liquidity_risk': liquidity_risk,
        'market_depth': market_depth,
        'time_to_sell_months': time_to_sell_months,
        'value_haircut_base': (base_haircut + zoning_addon),
        'holding_cost_rate_pa': holding_cost_rate_pa,
        'selling_cost_rate': selling_cost_rate,
        'contract_rate_proxy': contract_rate_proxy,
        'cost_of_funds_proxy': cost_of_funds_proxy,
        'discount_rate': discount_rate,
    })

land = pd.DataFrame(rows)

print('Land/Subdivision cases:', land.shape)
display(land['segment'].value_counts())
display(land['zoning_stage'].value_counts())
land.head()



## 2. Base LGD Logic


In [ ]:
def compute_land_lgd(df, haircut_addon=0.0, time_multiplier=1.0, holding_addon=0.0, rate_addon=0.0):
    x = df.copy()

    haircut = (x['value_haircut_base'] + haircut_addon + 0.08 * x['liquidity_risk'] + 0.06 * (1 - x['market_depth'])).clip(0.05, 0.70)
    time_months = (x['time_to_sell_months'] * time_multiplier).clip(4, 72)
    holding_rate = (x['holding_cost_rate_pa'] + holding_addon).clip(0.02, 0.18)
    disc_rate = (x['discount_rate'] + rate_addon).clip(0.03, 0.20)

    gross_sale = x['gross_land_value'] * (1 - haircut)
    holding_cost = x['gross_land_value'] * holding_rate * (time_months / 12.0)
    selling_cost = x['gross_land_value'] * x['selling_cost_rate']
    net_sale_pre_discount = (gross_sale - holding_cost - selling_cost).clip(lower=0.0)
    discount_factor = (1 + disc_rate) ** (time_months / 12.0)
    recovery_pv = net_sale_pre_discount / discount_factor
    lgd = ((x['ead'] - recovery_pv) / x['ead']).clip(0.0, 1.0)

    return lgd, time_months, recovery_pv

land['lgd_base'], land['time_to_sell_base'], land['recovery_pv_base'] = compute_land_lgd(land)

ewa_base = exposure_weighted_average(land, 'lgd_base', 'ead')
print(f'EAD-weighted base LGD: {ewa_base:.2%}')
display(land[['lgd_base', 'time_to_sell_base', 'recovery_pv_base']].describe().round(4))



## 3. Downturn / Stress LGD (Longer Recovery, Stronger Impact, and Final Overlay)

Downturn logic applies stronger value haircuts and longer sale horizons.  
A small explicit final overlay is then added so cross-product comparisons use a consistent `lgd_final` metric across secured products.


In [ ]:
scenario_specs = [
    {'scenario': 'base', 'haircut_addon': 0.00, 'time_multiplier': 1.00, 'holding_addon': 0.000, 'rate_addon': 0.000},
    {'scenario': 'mild_downturn', 'haircut_addon': 0.05, 'time_multiplier': 1.20, 'holding_addon': 0.010, 'rate_addon': 0.008},
    {'scenario': 'severe_downturn', 'haircut_addon': 0.12, 'time_multiplier': 1.45, 'holding_addon': 0.025, 'rate_addon': 0.020},
]

scenario_rows = []
for s in scenario_specs:
    lgd_s, time_s, rec_s = compute_land_lgd(
        land,
        haircut_addon=s['haircut_addon'],
        time_multiplier=s['time_multiplier'],
        holding_addon=s['holding_addon'],
        rate_addon=s['rate_addon'],
    )
    scenario = s['scenario']
    land[f'lgd_{scenario}'] = lgd_s
    land[f'time_to_sell_{scenario}'] = time_s
    land[f'recovery_pv_{scenario}'] = rec_s

    overlay_s = (
        0.012
        + 0.020 * land['liquidity_risk']
        + 0.012 * (1.0 - land['market_depth'])
        + 0.012 * ((time_s - 16.0) / 30.0).clip(0.0, 1.0)
    ).clip(0.012, 0.085)
    land[f'final_overlay_addon_{scenario}'] = overlay_s
    land[f'lgd_final_{scenario}'] = (lgd_s + overlay_s).clip(0.0, 1.0)

    scenario_rows.append({
        'scenario': scenario,
        'ead_weighted_lgd_base': exposure_weighted_average(land, f'lgd_{scenario}', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(land, f'lgd_{scenario}', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(land, f'lgd_final_{scenario}', 'ead'),
        'mean_time_to_sell_months': float(time_s.mean()),
    })

scenario_summary = pd.DataFrame(scenario_rows)
scenario_summary['mean_recovery_time_months'] = scenario_summary['mean_time_to_sell_months']
land['lgd_downturn'] = land['lgd_severe_downturn']
land['final_overlay_addon'] = land['final_overlay_addon_severe_downturn']
land['lgd_final'] = land['lgd_final_severe_downturn']

display(scenario_summary.round(4))



## 4. Weighted LGD Outputs by Segment and Zoning Stage


In [ ]:
segment_rows = []
for (seg, stage), grp in land.groupby(['segment', 'zoning_stage'], observed=True):
    segment_rows.append({
        'segment': seg,
        'zoning_stage': stage,
        'case_count': len(grp),
        'total_ead': grp['ead'].sum(),
        'ead_weighted_lgd_base': exposure_weighted_average(grp, 'lgd_base', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(grp, 'lgd_downturn', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(grp, 'lgd_final', 'ead'),
        'mean_liquidity_risk': grp['liquidity_risk'].mean(),
        'mean_market_depth': grp['market_depth'].mean(),
        'mean_time_to_sell_base': grp['time_to_sell_base'].mean(),
        'mean_value_haircut_base': grp['value_haircut_base'].mean(),
    })

segment_summary = pd.DataFrame(segment_rows).sort_values(
    ['ead_weighted_lgd_final', 'total_ead'], ascending=[False, False]
).reset_index(drop=True)

display(segment_summary.head(20).round(4))

chart_df = land.groupby('segment', observed=True).apply(
    lambda g: pd.Series({
        'ead_weighted_lgd_base': exposure_weighted_average(g, 'lgd_base', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(g, 'lgd_downturn', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final', 'ead'),
    }), include_groups=False
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = chart_df.set_index('segment')[['ead_weighted_lgd_base', 'ead_weighted_lgd_downturn', 'ead_weighted_lgd_final']]
plot_df.plot(kind='bar', ax=ax, color=['#4c72b0', '#dd8452', '#c44e52'])
ax.set_title('Land/Subdivision LGD by Segment (EAD-Weighted Base/Downturn/Final)')
ax.set_ylabel('LGD')
ax.set_xlabel('Segment')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'land_subdivision_weighted_lgd.png'), dpi=140, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'land_subdivision_weighted_lgd_comparison.png'), dpi=140, bbox_inches='tight')
plt.show()



## 5. Validation and Exports


In [ ]:
validation_checks = pd.DataFrame([
    {'check_name': 'lgd_base_between_0_and_1', 'passed': bool(land['lgd_base'].between(0, 1).all())},
    {'check_name': 'lgd_downturn_between_0_and_1', 'passed': bool(land['lgd_downturn'].between(0, 1).all())},
    {'check_name': 'lgd_final_between_0_and_1', 'passed': bool(land['lgd_final'].between(0, 1).all())},
    {'check_name': 'downturn_not_below_base', 'passed': bool((land['lgd_downturn'] >= land['lgd_base']).all())},
    {'check_name': 'final_not_below_downturn', 'passed': bool((land['lgd_final'] >= land['lgd_downturn']).all())},
    {'check_name': 'downturn_has_longer_sale_time', 'passed': bool((land['time_to_sell_severe_downturn'] >= land['time_to_sell_base']).all())},
])

display(validation_checks)

land.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'land_subdivision_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'land_subdivision_segment_summary.csv'), index=False)
scenario_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'land_subdivision_scenario_summary.csv'), index=False)
validation_checks.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'land_subdivision_validation_checks.csv'), index=False)

print('Saved land/subdivision outputs:')
print('- land_subdivision_loan_level_output.csv')
print('- land_subdivision_segment_summary.csv')
print('- land_subdivision_scenario_summary.csv')
print('- land_subdivision_validation_checks.csv')



## Assumptions and Limitations

- Dataset is synthetic and built for transparent portfolio-project demonstration.
- Land exposures are modelled as no-income assets (`no_income_asset_flag = 1`) with liquidation-led recovery.
- Zoning/stage, liquidity risk, market depth, time-to-sell, and value haircut are proxy drivers in absence of internal tape.
- Stronger downturn treatment reflects both deeper value haircut and longer sale horizon than base case.
